# Project Phase 3 Orchestrator Wrapper

Notebook wrapper for the prototype agentic application for Peloton Fitness

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd()
if (cwd / "prototype").exists():
    sys.path.insert(0, str(cwd))
elif (cwd / "Project_Phase_3" / "prototype").exists():
    sys.path.insert(0, str(cwd / "Project_Phase_3"))

from prototype.orchestrator import AgenticOrchestrator

orchestrator = AgenticOrchestrator()
print("Orchestrator initialized.")


## Declare ask function and helper render chart function to interact with the graph

In [ ]:
import json

def _render_chart_from_story_output(story_output):
    if not isinstance(story_output, dict):
        return False

    chart_spec = story_output.get("chart_spec")
    if not isinstance(chart_spec, dict):
        return False

    if chart_spec.get("library") != "plotly":
        print("[chart] Unsupported chart library in story_output.chart_spec")
        return False

    try:
        import plotly.graph_objects as go
    except Exception:
        print("[chart] Plotly is not installed; cannot render chart_spec.")
        return False

    data = chart_spec.get("data", [])
    layout = chart_spec.get("layout", {})
    try:
        fig = go.Figure(data=data, layout=layout)
        fig.show()
        return True
    except Exception as exc:
        print(f"[chart] Failed to render chart_spec: {exc}")
        return False


def ask(
    user_query: str,
    thread_id: str = "default",
    show_story_output: bool = False,
    debug: bool = False,
    render_chart: bool = True,
):
    out = orchestrator.invoke(user_query, thread_id=thread_id)
    print("USER:", user_query)
    print("THREAD:", thread_id)
    print("\nASSISTANT:")
    print(out["response"])
    print(f"\n[routed domain={out['active_domain']} story={out['active_story_id']}]")

    if debug:
        rm = out.get("router_metrics", {})
        print("\nrouter_reason:", out.get("router_reason"))
        print("continuation_score:", rm.get("continuation_score"))
        print("domain_scores:", rm.get("domain_scores"))
        print("margin:", rm.get("margin"))
        print("domain_selected_by:", rm.get("domain_selected_by"))
        print("story_selected_by:", rm.get("story_selected_by"))
        print("domain_guardrail_override:", rm.get("domain_guardrail_override"))

        so = out.get("story_output") or {}
        if isinstance(so, dict):
            ep = so.get("evidence_plan") or {}
            lr = so.get("llm_rationale") or {}
            if ep or lr:
                print("\nllm_story_debug:")
                print("planner_source:", ep.get("planner_source"))
                print("planner_confidence:", ep.get("planner_confidence"))
                print("llm_rationale_source:", lr.get("source"))
                print("llm_rationale_confidence:", lr.get("confidence"))

            plan_snapshot = so.get("plan_snapshot") or {}
            if so.get("planner_source") or plan_snapshot:
                print("\nplanning_debug:")
                print("planner_source:", so.get("planner_source"))
                print("planner_confidence:", so.get("planner_confidence", plan_snapshot.get("planner_confidence")))
                print("needs_clarification:", so.get("needs_clarification", plan_snapshot.get("needs_clarification")))
                print("requested_slot:", so.get("requested_slot", plan_snapshot.get("requested_slot")))

    if render_chart:
        _render_chart_from_story_output(out.get("story_output"))

    if show_story_output and out.get("story_output") is not None:
        print("\nstory_output:")
        try:
            print(json.dumps(out["story_output"], indent=2, default=str))
        except Exception:
            print(out["story_output"])

    return out





## Orchestrator Graph (LangGraph + Mermaid)
Top-level orchestrator graph visualization

In [ ]:
from IPython.display import Markdown, display, Image
from langchain_core.runnables.graph_mermaid import MermaidDrawMethod
import threading
import queue

def _render_mermaid_png_notebook_safe(graph_obj, label="Graph"):
    """Render Mermaid PNG in notebooks without tripping the running asyncio loop."""
    png = None
    q = queue.Queue()

    def _worker():
        try:
            q.put(("ok", graph_obj.draw_mermaid_png(draw_method=MermaidDrawMethod.API, max_retries=5, retry_delay=2.0)))
        except Exception as e:
            q.put(("err", e))

    t = threading.Thread(target=_worker, daemon=True)
    t.start()
    t.join(timeout=45)

    if t.is_alive():
        print(f"{label}: API render timed out; trying default renderer...")
    else:
        status, val = q.get()
        if status == "ok":
            png = val
        else:
            print(f"{label}: API render error: {val}")

    if png is None:
        try:
            png = graph_obj.draw_mermaid_png(max_retries=5, retry_delay=2.0)
        except Exception as e:
            print(f"{label}: default mermaid.ink render error: {e}")

    return png

mermaid = orchestrator.get_orchestrator_mermaid()
display(Markdown("```mermaid\n" + mermaid + "\n```"))

png = _render_mermaid_png_notebook_safe(orchestrator.get_orchestrator_graph().get_graph(), label="Orchestrator")
if png:
    display(Image(png))
else:
    print("PNG render unavailable; Mermaid source displayed above.")


## Membership Fraud Story Graph
Render the LangGraph for `mf_story_1` and run some checks.

In [ ]:
from IPython.display import Markdown, display, Image
from prototype.stories.membership_fraud_story1 import get_membership_fraud_story1_mermaid

mf_mermaid = get_membership_fraud_story1_mermaid()
display(Markdown("```mermaid\n" + mf_mermaid + "\n```"))

try:
    # Build an ephemeral graph image from the same definition used by the story module
    from prototype.stories.membership_fraud_story1 import _get_security_graph
    png = _get_security_graph().get_graph().draw_mermaid_png()
    display(Image(png))
except Exception as e:
    print("PNG render unavailable; Mermaid source displayed above.")
    print(e)


In [ ]:
thread_id = "mf_security_check"

print("1) Missing member_id on event-check intent -> should ask for member_id")
_ = ask("Can you check my most recent suspicious login?", thread_id=thread_id, debug=True)

print("\n2) Event-only with explicit last-7-days window -> should retrieve and expose fallback metadata")
out2 = ask("Check MB001 for suspicious logins in the last 7 days.", thread_id=thread_id, debug=True, show_story_output=True)
so2 = out2.get("story_output", {})
print("   intent_route:", so2.get("intent_route"))
print("   timeframe:", so2.get("timeframe"), "effective_timeframe:", so2.get("effective_timeframe"))
print("   fallback_applied:", so2.get("fallback_applied"), "fallback_reason:", so2.get("fallback_reason"))
print("   self_check_flags:", so2.get("self_check_flags"))

print("\n3) How-to only intent (no member_id) -> should skip DB path and use security help KB")
out3 = ask("How do I set up multi-factor authentication?", thread_id=thread_id, debug=True, show_story_output=True)
so3 = out3.get("story_output", {})
print("   intent_route:", so3.get("intent_route"))
print("   security_help_snippets:", len(so3.get("security_help_snippets", [])))
print("   self_check_flags:", so3.get("self_check_flags"))

print("\n4) Mixed intent (event + how-to) -> should return event response plus grounded help when available")
out4 = ask("For MB001, I do not recognize this login. How do I change my password?", thread_id=thread_id, debug=True, show_story_output=True)
so4 = out4.get("story_output", {})
print("   intent_route:", so4.get("intent_route"))
print("   security_help_snippets:", len(so4.get("security_help_snippets", [])))
print("   self_check_flags:", so4.get("self_check_flags"))

print("\n5) Event-only recognized follow-up -> should remain event path")
out5 = ask("I recognize this device now.", thread_id=thread_id, debug=True, show_story_output=True)
so5 = out5.get("story_output", {})
print("   intent_route:", so5.get("intent_route"))
print("   self_check_flags:", so5.get("self_check_flags"))

print("\n6) No-data event query -> should avoid event-claim hallucinations and show check flags")
out6 = ask("Check MB999 for suspicious logins in the last 7 days.", thread_id=thread_id, debug=True, show_story_output=True)
so6 = out6.get("story_output", {})
print("   intent_route:", so6.get("intent_route"))
print("   timeframe:", so6.get("timeframe"), "effective_timeframe:", so6.get("effective_timeframe"))
print("   fallback_applied:", so6.get("fallback_applied"), "fallback_reason:", so6.get("fallback_reason"))
print("   self_check_flags:", so6.get("self_check_flags"))


Render the LangGraph for `mf_story_2` and run some checks.

In [ ]:
from IPython.display import Markdown, display, Image
from prototype.stories.membership_fraud_story2 import get_membership_fraud_story2_mermaid, _get_membership_support_graph

mf2_mermaid = get_membership_fraud_story2_mermaid()
display(Markdown("```mermaid\n" + mf2_mermaid + "\n```"))

graph = _get_membership_support_graph().get_graph()
png = _render_mermaid_png_notebook_safe(graph, label="Membership Fraud Story 2")
if png:
    display(Image(png))
else:
    print("PNG render unavailable; Mermaid source displayed above.")


In [ ]:
thread_id = "mf_issue_triage_demo"

print("1) Login issue triage + high-confidence RAG")
_ = ask("I cannot login and need a password reset because I am locked out.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n2) Billing issue triage + queue routing")
_ = ask("I was charged twice and need a refund on my latest invoice.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n3) Renewal issue triage + queue routing")
_ = ask("My subscription renewal failed and my membership expired.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n4) Ambiguous issue -> should require human review")
_ = ask("Something is wrong with my account and I need help.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n5) Security-alert phrasing should route to mf_story_1 (not mf_story_2)")
_ = ask("I got a suspicious login alert from a new device in another location.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n6) This question should route to mf_story_2 and should be high confidence RAG, but low-relevance KB results should cause it to route to human instead")
_ = ask("My bill has a charge that I know is incorrect.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n7) Low-confidence short query should trigger LLM disambiguation path")
out7 = ask("Need refund.", thread_id=thread_id, debug=True, show_story_output=True)
so7 = out7.get("story_output", {})
print("   classification_source:", so7.get("classification_source"))
print("   issue_category:", so7.get("issue_category"))
print("   requires_human_review:", so7.get("requires_human_review"))
print("   audit_trace_len:", len(so7.get("audit_trace", [])))

print("\n8) Out-of-scope issue should route to human review with no RAG attempt")
out8 = ask("How do I change my profile photo?", thread_id=thread_id, debug=True, show_story_output=True)
so8 = out8.get("story_output", {})
print("   issue_category:", so8.get("issue_category"))
print("   requires_human_review:", so8.get("requires_human_review"))
print("   rag_attempted:", so8.get("rag_attempted"))
print("   rag_fallback_reason:", so8.get("rag_fallback_reason"))
print("   audit_trace_len:", len(so8.get("audit_trace", [])))

print("\n9) Quick audit trace sanity check from one high-confidence query")
out9 = ask("I cannot login and need a password reset because I am locked out.", thread_id=thread_id, debug=True, show_story_output=True)
so9 = out9.get("story_output", {})
trace9 = so9.get("audit_trace", [])
print("   audit_trace:", trace9)
print("   has_parse:", any("parse_request" in str(t) for t in trace9))
print("   has_classify:", any("classify_deterministic" in str(t) for t in trace9))
print("   has_route:", any("route_queue" in str(t) for t in trace9))
print("   has_retrieve:", any("retrieve_kb" in str(t) for t in trace9))
print("   has_guardrails:", any("guardrails" in str(t) for t in trace9))


Render the LangGraph for `mf_story_3` and run some checks.

In [ ]:
from IPython.display import Markdown, display, Image
from prototype.stories.membership_fraud_story3 import get_membership_fraud_story3_mermaid, _get_tier_fit_graph

mf3_mermaid = get_membership_fraud_story3_mermaid()
display(Markdown("```mermaid\n" + mf3_mermaid + "\n```"))

graph = _get_tier_fit_graph().get_graph()
png = _render_mermaid_png_notebook_safe(graph, label="Membership Fraud Story 3")
if png:
    display(Image(png))
else:
    print("PNG render unavailable; Mermaid source displayed above.")


In [ ]:
thread_id = "mf_tier_fit_demo"

print("1) Missing member_id -> should ask for member_id")
_ = ask("Can you analyze whether my membership tier is optimal?", thread_id=thread_id, debug=True)

print("\n2) Provide member_id + timeframe -> should route to mf_story_3 and evaluate tier fit")
_ = ask("MB017 over the last 6 months", thread_id=thread_id, debug=True, show_story_output=True)

print("\n3) Ask for benefit-sensitive recommendation -> should use evidence planner and feature breakdown")
_ = ask("Please include benefits usage context and explain upgrade/downgrade options.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n4) Follow-up what-if style phrasing in same thread")
_ = ask("If my usage stays similar, should I keep this tier?", thread_id=thread_id, debug=True, show_story_output=True)

print("\n5) Upgrade scenario example (expected decision=upgrade)")
_ = ask("My member ID is MB010. Based on usage over the last 6 months, is my membership tier still right?", thread_id=thread_id, debug=True, show_story_output=True)

print("\n6) Downgrade scenario example (expected decision=downgrade)")
_ = ask("My member ID is MB034. Based on my usage over the recent monthly average, should I downgrade my membership tier?", thread_id=thread_id, debug=True, show_story_output=True)


## Business Marketing Story Graphs
Render the LangGraph for `bm_story_1`, and run some checks.

In [ ]:
from IPython.display import Markdown, display, Image
from prototype.stories.business_marketing_story1 import get_business_marketing_story1_mermaid, _get_business_marketing_graph

bm1_mermaid = get_business_marketing_story1_mermaid()
display(Markdown("```mermaid\n" + bm1_mermaid + "\n```"))

try:
    png = _get_business_marketing_graph().get_graph().draw_mermaid_png()
    display(Image(png))
except Exception as e:
    print("PNG render unavailable; Mermaid source displayed above.")
    print(e)


In [ ]:
thread_id = "bm_feedback_check"

tests = [
    (
        "1) Normal baseline",
        "Summarize campaign feedback themes in the last 4 weeks and suggest 3 content adjustments",
        True,
    ),
    (
        "2) Filtered query (campaign + channel + short window)",
        "Summarize last week feedback for CAMP105 from email and suggest adjustments",
        False,
    ),
    (
        "3) User already broad timeframe (>=12 weeks)",
        "Summarize last 12 weeks feedback for CAMP105 from social and suggest adjustments",
        False,
    ),
    (
        "4) Likely sparse/no-match constraints",
        "Summarize last 12 weeks feedback for member MB999 for CAMP999999 from social and suggest adjustments",
        True,
    ),
]

for label, query, show_output in tests:
    print(f"\n{label}")
    print(f"Query: {query}")
    _ = ask(
        query,
        thread_id=thread_id,
        debug=True,
        show_story_output=show_output,
    )



Render the LangGraph for `bm_story_2`, and run some checks.

In [ ]:
from IPython.display import Markdown, display, Image
from langchain_core.runnables.graph_mermaid import MermaidDrawMethod
from prototype.stories.business_marketing_story2 import get_business_marketing_story2_mermaid, _get_business_marketing_story2_graph

bm2_mermaid = get_business_marketing_story2_mermaid()
display(Markdown("```mermaid\n" + bm2_mermaid + "\n```"))

graph = _get_business_marketing_story2_graph().get_graph()
png = _render_mermaid_png_notebook_safe(graph, label="Business Marketing Story 2")
if png:
    display(Image(png))
else:
    print("PNG render unavailable; Mermaid source displayed above.")


In [ ]:
thread_id = "bm_kpi_check"

print("1) Clarification gate on ambiguous request")
_ = ask("Can you help with campaign performance?", thread_id=thread_id, debug=True, show_story_output=True)

print("\n2) Clarification follow-up + resolved analysis")
_ = ask("Compare ROAS and CAC by channel for last 4 weeks", thread_id=thread_id, debug=True, show_story_output=True)

print("\n3) Multi-turn memory carry-forward (same scope)")
_ = ask("and keep same scope", thread_id=thread_id, debug=True, show_story_output=True)

print("\n4) Explicit override should beat memory")
_ = ask("Only show underperformers by campaign_id", thread_id=thread_id, debug=True, show_story_output=True)

print("\n5) Definitions intent")
_ = ask("What are CTR, CAC, and ROAS?", thread_id=thread_id, debug=True, show_story_output=True)

print("\n6) Critic/replan behavior on sparse scope")
_ = ask("Compare campaign metrics for CAMP999 by channel in the latest week", thread_id=thread_id, debug=True, show_story_output=True)

print("\n7) Unsupported-dimension self-check fallback")
_ = ask("Show weekly campaign performance by geography for last month", thread_id=thread_id, debug=True, show_story_output=True)


Render the LangGraph for `bm_story_3`, and run some checks.

In [ ]:
from IPython.display import Markdown, display, Image
from prototype.stories.business_marketing_story3 import get_business_marketing_story3_mermaid, _get_business_marketing_story3_graph

bm3_mermaid = get_business_marketing_story3_mermaid()
display(Markdown("```mermaid\n" + bm3_mermaid + "\n```"))

try:
    png = _get_business_marketing_story3_graph().get_graph().draw_mermaid_png()
    display(Image(png))
except Exception as e:
    print("PNG render unavailable; Mermaid source displayed above.")
    print(e)


In [ ]:
thread_id = "bm_leads_check"

print("1) Default assumptions (recent + email) with top-N")
_ = ask("Generate prioritized leads and draft follow-up messages for top 5", thread_id=thread_id, debug=True, show_story_output=True)

print("\n2) Explicit timeframe + channel + interest filter")
_ = ask("For last 14 days, generate top 6 cycling leads and draft call follow-ups in a friendly tone", thread_id=thread_id, debug=True, show_story_output=True)

print("\n3) Suppression guardrail check (email channel)")
_ = ask("Show top 12 leads for last 7 days and draft email outreach", thread_id=thread_id, debug=True, show_story_output=True)

In [ ]:
# Multi-turn ambiguity resolution check in one thread
thread_id = "bm_story3_multiturn_resolution"

print("Turn 1: ambiguous")
t1 = ask("Generate top leads and draft follow-ups", thread_id=thread_id, debug=True, show_story_output=True)

print("\nTurn 2: resolve ambiguity explicitly")
t2 = ask("Use last 14 days and email channel, focus on cycling", thread_id=thread_id, debug=True, show_story_output=True)

print("\nTurn 3: refine only tone/top_n")
t3 = ask("Make tone consultative and show top 5", thread_id=thread_id, debug=True, show_story_output=True)

# Expected: stays in business_marketing / bm_story_3 across turns
for i, t in enumerate([t1, t2, t3], start=1):
    assert t["active_domain"] == "business_marketing", f"turn {i}: wrong domain"
    assert t["active_story_id"] == "bm_story_3", f"turn {i}: wrong story"

print("Multi-turn routing checks passed.")


## Data Science Story Graphs
Render the LangGraph for `ds_story_1` and run some checks.

In [ ]:
from IPython.display import Markdown, display, Image
from prototype.stories.data_science_story1 import get_data_science_story1_mermaid, _get_data_science_story1_graph
ds1_mermaid = get_data_science_story1_mermaid()
display(Markdown("```mermaid\n" + ds1_mermaid + "\n```"))
try:
    png = _get_data_science_story1_graph().get_graph().draw_mermaid_png()
    display(Image(png))
except Exception as e:
    print("PNG render unavailable; Mermaid source displayed above.")
    print(e)


Run these to test `ds_story_1` chart-building behavior and auto-rendered Plotly charts from `story_output.chart_spec`.

In [ ]:
thread_id = "ds_viz_story1_check"

print("1) Member-scoped trend line chart")
_ = ask("For MB001, create a weekly trend chart of duration over the last 8 weeks.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n2) Bar chart by workout type")
_ = ask("For MB001, show a bar chart of average calories by workout type over the last 8 weeks.", thread_id=thread_id, debug=True)

print("\n3) Scatter chart for metric relationship")
_ = ask("For MB001, create a scatter plot of output_kj versus calories in the last 8 weeks.", thread_id=thread_id, debug=True)

print("\n4) Histogram distribution chart")
_ = ask("For MB001, plot a histogram of strive score for the last 8 weeks.", thread_id=thread_id, debug=True)

print("\n5) Box plot by class type")
_ = ask("For MB001, make a box plot of duration by workout type over the last 8 weeks.", thread_id=thread_id, debug=True)

Use one thread to validate carry-forward refinement in `ds_story_1`

In [ ]:
thread_id = "ds_viz_refinement_check_"

print("1) Initial ambiguous request -> should develop a plan of possible visualizations and select the best one.")
_ = ask("Show me a chart of my progress for MB001", thread_id=thread_id, debug=True, show_story_output=True)

print("\n2) Refinement: switch chart type and metric relation")
_ = ask("Switch to scatter with calories vs output_kj", thread_id=thread_id, debug=True, show_story_output=True)

print("\n3) Refinement: keep same chart, change timeframe")
_ = ask("Keep the same chart but use the last 12 weeks", thread_id=thread_id, debug=True, show_story_output=True)

print("\n4) Refinement: switch segmentation")
_ = ask("Now segment by weekday instead", thread_id=thread_id, debug=True, show_story_output=True)

print("\n5) Provides clarification of domain")
_ = ask("Data science: Can we look at the same information grouped by weekday?", thread_id=thread_id, debug=True, show_story_output=True)

Render the LangGraph for `ds_story_2` and run some checks.

In [ ]:
from IPython.display import Markdown, display, Image
from prototype.stories.data_science_story2 import get_data_science_story2_mermaid, _get_data_science_graph

ds_mermaid = get_data_science_story2_mermaid()
display(Markdown("```mermaid\n" + ds_mermaid + "\n```"))

try:
    png = _get_data_science_graph().get_graph().draw_mermaid_png()
    display(Image(png))
except Exception as e:
    print("PNG render unavailable; Mermaid source displayed above.")
    print(e)


In [ ]:
thread_id = "ds_workout_trends_check"

print("1) Missing member_id -> should ask for member_id")
_ = ask("Am I improving over the past 8 weeks?", thread_id=thread_id, debug=True, show_story_output=True)

print("\n2) Provide member + trend question")
_ = ask("For MB001, am I improving in my workouts over the last 8 weeks?", thread_id=thread_id, debug=True, show_story_output=True)

print("\n3) Drivers + intensity question")
_ = ask("For MB001 what is driving intensity and do weekdays differ?", thread_id=thread_id, debug=True, show_story_output=True)

print("\n4) Anomaly question")
_ = ask("For MB001, any unusual drops or spikes in the last 8 weeks?", thread_id=thread_id, debug=True, show_story_output=True)


Render the LangGraph for `ds_story_3` and run some checks.

In [ ]:
from IPython.display import Markdown, display, Image
from prototype.stories.data_science_story3 import get_data_science_story3_mermaid, _get_peer_benchmark_graph

ds3_mermaid = get_data_science_story3_mermaid()
display(Markdown("```mermaid\n" + ds3_mermaid + "\n```"))

try:
    png = _get_peer_benchmark_graph().get_graph().draw_mermaid_png()
    display(Image(png))
except Exception as e:
    print("PNG render unavailable; Mermaid source displayed above.")
    print(e)


In [ ]:
thread_id = "ds_peer_benchmark_check"

print("1) Generic metrics request -> should ask which metrics")
_ = ask("Compare MB001 metrics to peers.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n2) Clarify metrics -> should proceed with selected metrics")
_ = ask("Use weekly workouts and consistency.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n3) Ask for custom peer definition")
_ = ask("Now compare me to peers with similar activity level over the last 12 weeks.", thread_id=thread_id, debug=True, show_story_output=True)

In [ ]:
thread_id = "ds_peer_benchmark_high_uncertainty"

print("1) Vague cohort + vague metrics")
_ = ask("Can you benchmark me against people like me and tell me what to fix? My member ID is MB001.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n2) Generic metrics wording")
_ = ask("How am I doing versus peers? Use the right metrics.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n3) Underspecified cohort constraints")
_ = ask("Compare me to others but not everyone, and focus on what matters most.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n4) Ambiguous but intentful peer comparison")
_ = ask("I want a fair peer comparison for my workouts and advice from that.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n5) Similar-user phrasing + vague timeframe")
_ = ask("Show where I stand vs similar users recently.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n6) Broad request without slots")
_ = ask("Do a peer check for me and recommend next steps.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n7) Hedged metric selection")
_ = ask("Compare my performance with peers, maybe by consistency or something.", thread_id=thread_id, debug=True, show_story_output=True)

print("\n8) Cohort mention without definition")
_ = ask("What should I improve compared to others in my cohort?", thread_id=thread_id, debug=True, show_story_output=True)


## Interactive Multi-Turn Loop
Run this cell for live manual testing. Type `exit` or `quit` to stop.

In [ ]:
active_thread_id = input("Thread ID (default=user_live): " ).strip() or "user_live"
print(f"Interactive multi-turn mode started for thread_id={active_thread_id}. Type 'exit' to stop.")
while True:
    user_text = input("You: " ).strip()
    if user_text.lower() in {"exit", "quit"}:
        print("Stopped interactive mode.")
        break
    if not user_text:
        continue

    out = ask(user_text, thread_id=active_thread_id)
    print(f"(thread={active_thread_id}, domain={out['active_domain']}, story={out['active_story_id']})")
    print('-' * 80)

In [ ]:
active_thread_id = input("Thread ID (default=user_live): " ).strip() or "user_live"
print(f"Interactive multi-turn mode started for thread_id={active_thread_id}. Type 'exit' to stop.")
while True:
    user_text = input("You: " ).strip()
    if user_text.lower() in {"exit", "quit"}:
        print("Stopped interactive mode.")
        break
    if not user_text:
        continue

    out = ask(user_text, thread_id=active_thread_id, debug=True)
    print("pending_slot_type:", out["state"]["pending_slot_type"])
    print("member:", out["state"]["member"])

    print(f"(thread={active_thread_id}, domain={out['active_domain']}, story={out['active_story_id']})")
    print('-' * 80)

## Inspect state

In [ ]:
orchestrator.state